In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams["axes.prop_cycle"] = plt.cycler(color=["#0F766E","#E4572E","#6C4AB6","#F2A900"])

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 20)

In [17]:
# from google.colab import files
# uploaded = files.upload()
# file_name = list(uploaded.keys())[0] 

In [21]:
file_name = "MatchResultsrefined.xlsx"   # full path to your file

bat = pd.read_excel(file_name, sheet_name="Batting")
team_bat = pd.read_excel(file_name, sheet_name="TeamBat")
bowl = pd.read_excel(file_name, sheet_name="Bowling")

print("Batting shape:", bat.shape)
print("TeamBat shape:", team_bat.shape)
print("Bowling shape:", bowl.shape)

ImportError: Missing optional dependency 'openpyxl'.  Use pip or conda to install openpyxl.

In [ ]:
# Convert dates
bat["Date"] = pd.to_datetime(bat["Date"], errors="coerce")
team_bat["Date"] = pd.to_datetime(team_bat["Date"], errors="coerce")
bowl["Date"] = pd.to_datetime(bowl["Date"], errors="coerce")

# Convert numeric columns
bat_numeric = ["R", "B", "M", "4s", "6s", "SR", "Chase", "Win",
               "BoundaryRate", "RunMint"]

bowl_numeric = ["O", "M", "R", "W", "Econ", "Dots", "4s", "6s",
                "Wide", "NB", "Win", "Potential"]

for col in bat_numeric:
    bat[col] = pd.to_numeric(bat[col], errors="coerce").fillna(0)

for col in bowl_numeric:
    bowl[col] = pd.to_numeric(bowl[col], errors="coerce").fillna(0)

print("Data cleaned successfully!")

Question 1:  Who are the top 10 batsmen based on their total runs scored across all matches?

In [ ]:
top_10_runs = (
    bat.groupby("Player")
    .agg(
        Total_Runs=("R", "sum"),
        Balls_Faced=("B", "sum"),
        Fours=("4s", "sum"),
        Sixes=("6s", "sum")
    )
    .sort_values("Total_Runs", ascending=False)
    .head(10)
    .reset_index()
)

top_10_runs.style.background_gradient(cmap="Purples", subset=["Total_Runs"])

### Question 2: Which 10 players have the highest strike rate among players who have faced at least 50 balls?

In [ ]:
player_batting = (
    bat.groupby("Player")
    .agg(
        Runs=("R", "sum"),
        Balls=("B", "sum"),
        Fours=("4s", "sum"),
        Sixes=("6s", "sum")
    )
)

player_batting["Strike_Rate"] = (
    player_batting["Runs"] / player_batting["Balls"] * 100
)

top_10_strike_rate = (
    player_batting[player_batting["Balls"] >= 50]
    .sort_values("Strike_Rate", ascending=False)
    .head(10)
    .reset_index()
)

top_10_strike_rate.style.background_gradient(cmap="YlGn", subset=["Strike_Rate"])

### Question 3:Which player has scored the most fours and which player has scored the most sixes?

In [ ]:
boundary_summary = (
    bat.groupby("Player")
    .agg(
        Fours=("4s", "sum"),
        Sixes=("6s", "sum")
    )
    .sort_values(["Fours", "Sixes"], ascending=False)
)

most_fours = boundary_summary["Fours"].idxmax()
most_sixes = boundary_summary["Sixes"].idxmax()

print("Most fours:", most_fours)
print("Number of fours:", boundary_summary.loc[most_fours, "Fours"])

print("Most sixes:", most_sixes)
print("Number of sixes:", boundary_summary.loc[most_sixes, "Sixes"])

### Question 4: Which team has won the highest number of matches, and how many matches did they win?

In [ ]:
matches = bat[["Date", "Team", "Vs", "Win"]].drop_duplicates()

team_wins = (
    matches.groupby("Team")["Win"]
    .sum()
    .sort_values(ascending=False)
)

team_wins.head(10)

### Question 5: For each match, who was the highest run scorer? Can you display the player name, team, opponent, match date, and runs scored?

In [ ]:
match_top_scorers = (
    bat.sort_values("R", ascending=False)
    .groupby(["Date", "Team", "Vs"], as_index=False)
    .first()
)

match_top_scorers = match_top_scorers[
    ["Date", "Team", "Vs", "Player", "R"]
]

match_top_scorers = match_top_scorers.rename(
    columns={
        "Date": "Match_Date",
        "Vs": "Opponent",
        "R": "Runs_Scored"
    }
)

match_top_scorers

In [ ]:
display(match_top_scorers.to_string(index=False))

### Question 6: Do teams perform better while chasing or while batting first? Compare their average runs and overall performance in both situations

In [ ]:
chasing_analysis = (
    bat.groupby("Chase")
    .agg(
        Average_Runs=("R", "mean"),
        Total_Runs=("R", "sum"),
        Innings=("R", "count"),
        Wins=("Win", "sum")
    )
)

chasing_analysis["Win_Percentage"] = (
    chasing_analysis["Wins"] /
    chasing_analysis["Innings"] * 100
)

chasing_analysis.index = [
    "Batting First" if x == 0 else "Chasing"
    for x in chasing_analysis.index
]

chasing_analysis

### Question 7: Who are the top 10 bowlers based on total wickets taken, and what are their average economy rates?

In [ ]:
bowler_summary = (
    bowl.groupby("Bowler")
    .agg(
        Total_Wickets=("W", "sum"),
        Total_Overs=("O", "sum"),
        Runs_Conceded=("R", "sum")
    )
)

bowler_summary["Average_Economy"] = (
    bowler_summary["Runs_Conceded"] /
    bowler_summary["Total_Overs"] * 6
)

top_10_bowlers = (
    bowler_summary
    .sort_values(
        ["Total_Wickets", "Average_Economy"],
        ascending=[False, True]
    )
    .head(10)
    .reset_index()
)

top_10_bowlers.style.background_gradient(cmap="Oranges", subset=["Total_Wickets"])

### Question 8: Which players have scored the most runs through boundaries (fours and sixes), and how does their strike rate compare with other players

In [ ]:
boundary_players = (
    bat.groupby("Player")
    .agg(
        Runs=("R", "sum"),
        Balls=("B", "sum"),
        Fours=("4s", "sum"),
        Sixes=("6s", "sum")
    )
)

boundary_players["Boundary_Runs"] = (
    boundary_players["Fours"] * 4 +
    boundary_players["Sixes"] * 6
)

boundary_players["Strike_Rate"] = (
    boundary_players["Runs"] /
    boundary_players["Balls"] * 100
)

top_boundary_players = (
    boundary_players
    .sort_values("Boundary_Runs", ascending=False)
    .head(10)
    .reset_index()
)

top_boundary_players.style.background_gradient(cmap="PuRd", subset=["Boundary_Runs"])

### Question 9:Can you create a Player Performance Score using runs, strike rate, fours, sixes, and balls faced, and use it to rank the top 20 batsmen?

In [ ]:
performance = (
    bat.groupby("Player")
    .agg(
        Runs=("R", "sum"),
        Balls=("B", "sum"),
        Fours=("4s", "sum"),
        Sixes=("6s", "sum")
    )
)

performance["Strike_Rate"] = (
    performance["Runs"] /
    performance["Balls"] * 100
)

# Normalize each component
performance["Runs_N"] = performance["Runs"] / performance["Runs"].max()
performance["SR_N"] = performance["Strike_Rate"] / performance["Strike_Rate"].max()
performance["Fours_N"] = performance["Fours"] / performance["Fours"].max()
performance["Sixes_N"] = performance["Sixes"] / performance["Sixes"].max()
performance["Balls_N"] = performance["Balls"] / performance["Balls"].max()

# Calculate Player Performance Score
performance["PPS"] = 100 * (
    0.35 * performance["Runs_N"] +
    0.25 * performance["SR_N"] +
    0.15 * performance["Fours_N"] +
    0.15 * performance["Sixes_N"] +
    0.10 * performance["Balls_N"]
)

top_20_batsmen = (
    performance
    .sort_values("PPS", ascending=False)
    .head(20)
    .reset_index()
)

top_20_batsmen[
    ["Player", "Runs", "Balls", "Fours", "Sixes",
     "Strike_Rate", "PPS"]
].style.background_gradient(cmap="YlOrBr", subset=["PPS"])

### Question 10:  Can you combine the batting and bowling data to identify the top 10 all-round performers? Create your own Match Impact Score using runs, wickets, strike rate, economy rate, boundaries, and match result?

In [ ]:
# -----------------------------
# Batting summary
# -----------------------------

bat_summary = (
    bat.groupby("Player")
    .agg(
        Runs=("R", "sum"),
        Balls=("B", "sum"),
        Fours=("4s", "sum"),
        Sixes=("6s", "sum"),
        Batting_Wins=("Win", "sum")
    )
)

bat_summary["Strike_Rate"] = (
    bat_summary["Runs"] /
    bat_summary["Balls"] * 100
)

# -----------------------------
# Bowling summary
# -----------------------------

bowl_summary = (
    bowl.groupby("Bowler")
    .agg(
        Wickets=("W", "sum"),
        Overs=("O", "sum"),
        Runs_Conceded=("R", "sum"),
        Bowling_Wins=("Win", "sum")
    )
)

bowl_summary["Economy_Rate"] = (
    bowl_summary["Runs_Conceded"] /
    bowl_summary["Overs"] * 6
)

# Rename Bowler to Player
bowl_summary = bowl_summary.reset_index()
bowl_summary = bowl_summary.rename(columns={"Bowler": "Player"})
bowl_summary = bowl_summary.set_index("Player")

# -----------------------------
# Combine batting and bowling
# -----------------------------

all_rounders = bat_summary.join(
    bowl_summary,
    how="outer",
    lsuffix="_Bat",
    rsuffix="_Bowl"
).fillna(0)

all_rounders.head()

In [ ]:
# Normalize batting factors
all_rounders["Runs_N"] = (
    all_rounders["Runs"] /
    all_rounders["Runs"].max()
)

all_rounders["SR_N"] = (
    all_rounders["Strike_Rate"] /
    all_rounders["Strike_Rate"].max()
)

all_rounders["Fours_N"] = (
    all_rounders["Fours"] /
    all_rounders["Fours"].max()
)

all_rounders["Sixes_N"] = (
    all_rounders["Sixes"] /
    all_rounders["Sixes"].max()
)

# Normalize wickets
all_rounders["Wickets_N"] = (
    all_rounders["Wickets"] /
    all_rounders["Wickets"].max()
)

# Normalize economy: lower is better
all_rounders["Economy_N"] = (
    all_rounders["Economy_Rate"] /
    all_rounders["Economy_Rate"].max()
)

all_rounders["Economy_Score"] = (
    1 - all_rounders["Economy_N"]
)

# Normalize match wins
all_rounders["Wins_N"] = (
    all_rounders["Batting_Wins"] /
    all_rounders["Batting_Wins"].max()
)

In [ ]:
all_rounders["Match_Impact_Score"] = 100 * (
    0.30 * all_rounders["Runs_N"] +
    0.15 * all_rounders["SR_N"] +
    0.10 * all_rounders["Fours_N"] +
    0.10 * all_rounders["Sixes_N"] +
    0.20 * all_rounders["Wickets_N"] +
    0.10 * all_rounders["Economy_Score"] +
    0.05 * all_rounders["Wins_N"]
)

top_10_all_rounders = (
    all_rounders
    .sort_values("Match_Impact_Score", ascending=False)
    .head(10)
    .reset_index()
)

top_10_all_rounders[
    ["Player", "Runs", "Strike_Rate", "Fours", "Sixes",
     "Wickets", "Economy_Rate", "Match_Impact_Score"]
].style.background_gradient(cmap="BuGn", subset=["Match_Impact_Score"])

In [ ]:
all_rounders[
    (all_rounders["Runs"] > 0) &
    (all_rounders["Wickets"] > 0)
]

### Chart 1 - Top 10 run scores

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    top_10_runs["Player"],
    top_10_runs["Total_Runs"],
    color="#6C4AB6",
    edgecolor="black"
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Player")
plt.ylabel("Total Runs")
plt.title("Top 10 Batsmen by Total Runs")
plt.tight_layout()
plt.show()

### Chart 2 - Top 10 bowlers by wickets

In [ ]:
plt.figure(figsize=(12, 6))

plt.bar(
    top_10_bowlers["Bowler"],
    top_10_bowlers["Total_Wickets"],
    color="#E4572E",
    edgecolor="black"
)

plt.xticks(rotation=45, ha="right")
plt.xlabel("Bowler")
plt.ylabel("Total Wickets")
plt.title("Top 10 Bowlers by Total Wickets")
plt.tight_layout()
plt.show()

### Chart 3-- Batting First vs Chasing

In [ ]:
plt.figure(figsize=(7, 5))

plt.bar(
    chasing_analysis.index,
    chasing_analysis["Average_Runs"],
    color=["#0F766E", "#F2A900"],
    edgecolor="black"
)

plt.xlabel("Batting Situation")
plt.ylabel("Average Runs")
plt.title("Average Runs: Batting First vs Chasing")
plt.show()